## 1. Installation et imports

In [ ]:
# !pip install -q -U transformers accelerate bitsandbytes pandas numpy tqdm

In [ ]:
!pip install --upgrade scipy scikit-learn numba transformers

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pandas numpy tqdm

In [ ]:
import os, re, json, math, time, random, hashlib, unicodedata, textwrap, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

pd.set_option("display.max_colwidth", 200)
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)

## 2. Configuration

Paramètres importants  :
- **Décodage glouton** (`do_sample=False`) : réponses reproductibles, indispensables pour comparer. Les trois configurations devront utiliser *les mêmes* paramètres de génération.
- `MAX_NEW_TOKENS` : coupe les réponses trop longues.
- `BATCH_SIZE` : à baisser en cas de `CUDA out of memory`.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
USE_4BIT = True            # Quantification NF4. Gardée même si la VRAM permettrait mieux : voir note en section 4.
SEED = 42
MAX_NEW_TOKENS = 300       # réponses du modèle évalué
JUDGE_MAX_NEW_TOKENS = 40  # sortie JSON du juge
BATCH_SIZE = 4

TEST_SET_PATH = Path("test_set_b1.jsonl")      # à placer à côté du notebook
OUTPUT_DIR = Path("outputs/step1_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ABSTAIN_MARKER = "INFORMATION ABSENTE"        # chaîne d'abstention imposée dans le prompt (réutilisée en B et C)

random.seed(SEED); np.random.seed(SEED); set_seed(SEED)

## 3. Vérification du matériel

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU CUDA détecté. Ce notebook utilise transformers + bitsandbytes (NVIDIA uniquement). "
        "Alternatives : Colab/Kaggle avec GPU, ou une chaîne llama.cpp (GGUF) / MLX à mettre en place séparément."
    )

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if bf16_ok else torch.float16   # T4 : pas de bf16 -> float16

print(f"GPU : {props.name} | VRAM : {vram_gb:.1f} Go | bf16 : {bf16_ok} | dtype de calcul : {COMPUTE_DTYPE}")
if vram_gb < 7:
    print("ATTENTION : moins de 7 Go de VRAM, le chargement 4-bit risque d'échouer ou de déborder sur le CPU.")